# NB06: Cross-Validation with Pangenome & Literature

Validate cultivation-bias signals by comparing against:
1. eggNOG/GapMind annotations for 147 ENIGMA genomes linked to `kbase_ke_pangenome`
2. Published ORFRC metagenome studies

**Requires**: NB04 outputs, BERDL Spark access

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from berdl_notebook_utils.setup_spark_session import get_spark_session

spark = get_spark_session()
DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Identify Pangenome-Linked ENIGMA Genomes

In [2]:
# Match ENIGMA cultured genomes to kbase_ke_pangenome via assembly accession.
# external_id format observed: "NCBI:GCF_*" (RefSeq) or "NCBI:GCA_*" (GenBank).
# Pangenome genome_id format: "RS_GCF_*" or "GB_GCA_*".

cult_meta = pd.read_csv(DATA_DIR / 'cultured_genome_metadata.tsv', sep='\t')

def extract_pangenome_id(ext_id):
    if pd.isna(ext_id):
        return None
    s = str(ext_id)
    if s.startswith('NCBI:'):
        s = s[5:]
    if s.startswith('GCF_'):
        return f'RS_{s}'
    if s.startswith('GCA_'):
        return f'GB_{s}'
    return None

cult_meta['pangenome_id'] = cult_meta['external_id'].apply(extract_pangenome_id)
linked = cult_meta[cult_meta['pangenome_id'].notna()].copy()
print(f'{len(linked)} ENIGMA genomes have GCF/GCA accessions (candidate pangenome IDs)')

candidates = sorted(set(linked['pangenome_id']))
acc_str = ','.join([f"'{a}'" for a in candidates])
pangenome_match = spark.sql(f"""
    SELECT genome_id
    FROM kbase_ke_pangenome.genome
    WHERE genome_id IN ({acc_str})
""").toPandas()

matched_pids = set(pangenome_match['genome_id'])
matched = linked[linked['pangenome_id'].isin(matched_pids)].copy()
matched = matched.rename(columns={'genome_id': 'enigma_genome_id'})
print(f'{len(matched)} ENIGMA-pangenome pairs matched ({len(matched_pids)} unique pangenome genomes)')

424 ENIGMA genomes have GCF/GCA accessions (candidate pangenome IDs)


153 ENIGMA-pangenome pairs matched (147 unique pangenome genomes)


## 2. eggNOG Annotations for Linked Genomes

In [3]:
# Pull eggNOG annotations for all genes of the matched pangenome genomes.
# Schema note: eggnog_mapper_annotations.query_name == gene.gene_id (e.g., NZ_XXX_n);
# columns are case-sensitive: KEGG_ko, COG_category, Description.

eggnog_long = pd.DataFrame()
if len(matched) > 0:
    pgids = sorted(set(matched['pangenome_id']))
    pgids_str = ','.join([f"'{p}'" for p in pgids])

    gene_eggnog = spark.sql(f"""
        SELECT g.genome_id, g.gene_id,
               e.KEGG_ko, e.COG_category, e.Description
        FROM kbase_ke_pangenome.gene g
        LEFT JOIN kbase_ke_pangenome.eggnog_mapper_annotations e
          ON g.gene_id = e.query_name
        WHERE g.genome_id IN ({pgids_str})
          AND e.KEGG_ko IS NOT NULL AND e.KEGG_ko != '' AND e.KEGG_ko != '-'
    """).toPandas()
    print(f'{len(gene_eggnog)} gene-level eggNOG annotations with KO assignments')

    def parse_kos(s):
        if not isinstance(s, str):
            return []
        return [x.replace('ko:', '').strip() for x in s.split(',') if x.strip() and x.strip() != '-']

    gene_eggnog['ko_list'] = gene_eggnog['KEGG_ko'].apply(parse_kos)
    eggnog_long = gene_eggnog.explode('ko_list').rename(columns={'ko_list': 'ko_id'})
    eggnog_long = eggnog_long[eggnog_long['ko_id'].notna() & (eggnog_long['ko_id'] != '')].copy()

    print(f'{eggnog_long["ko_id"].nunique()} unique KOs from eggNOG across {len(pgids)} matched genomes')

    cult_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
    matched_enigma_ids = set(matched['enigma_genome_id'])
    depot_kos  = set(cult_ko[cult_ko['genome_id'].isin(matched_enigma_ids)]['ko_id'].unique())
    eggnog_kos = set(eggnog_long['ko_id'].unique())

    print(f'\nKO sets (pooled over matched genomes):')
    print(f'  Depot KOs : {len(depot_kos)}')
    print(f'  eggNOG KOs: {len(eggnog_kos)}')
    print(f'  Shared    : {len(depot_kos & eggnog_kos)}')
    print(f'  Depot-only: {len(depot_kos - eggnog_kos)}')
    print(f'  eggNOG-only: {len(eggnog_kos - depot_kos)}')
else:
    print('No pangenome-linked genomes matched')

71760 gene-level eggNOG annotations with KO assignments
5899 unique KOs from eggNOG across 147 matched genomes



KO sets (pooled over matched genomes):
  Depot KOs : 7256
  eggNOG KOs: 5899
  Shared    : 5751
  Depot-only: 1505
  eggNOG-only: 148


## 3. Annotation Concordance

In [4]:
# Per-genome Jaccard concordance between Genome Depot KOs and eggNOG KOs.
if len(matched) > 0 and len(eggnog_long) > 0:
    cult_ko_idx = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
    depot_by_genome = (cult_ko_idx.groupby('genome_id')['ko_id']
                                   .apply(lambda s: set(s.unique())).to_dict())
    eggnog_by_genome = (eggnog_long.groupby('genome_id')['ko_id']
                                    .apply(lambda s: set(s.unique())).to_dict())

    rows = []
    for _, r in matched.iterrows():
        depot_set  = depot_by_genome.get(r['enigma_genome_id'], set())
        eggnog_set = eggnog_by_genome.get(r['pangenome_id'], set())
        union = depot_set | eggnog_set
        jac   = (len(depot_set & eggnog_set) / len(union)) if union else 0.0
        rows.append({
            'enigma_id':    r['enigma_genome_id'],
            'pangenome_id': r['pangenome_id'],
            'depot_kos':    len(depot_set),
            'eggnog_kos':   len(eggnog_set),
            'shared':       len(depot_set & eggnog_set),
            'jaccard':      jac,
        })
    jac_df = pd.DataFrame(rows)
    jac_df.to_csv(DATA_DIR / 'pangenome_crossvalidation.tsv', sep='\t', index=False)

    print(f'Per-genome concordance across {len(jac_df)} matched ENIGMA<->pangenome pairs:')
    print(f'  Median Jaccard: {jac_df["jaccard"].median():.3f}')
    print(f'  Mean   Jaccard: {jac_df["jaccard"].mean():.3f}')
    print(f'  IQR           : {jac_df["jaccard"].quantile(0.25):.3f} - {jac_df["jaccard"].quantile(0.75):.3f}')
    print()
    print('Top 10 by Jaccard:')
    print(jac_df.nlargest(10, 'jaccard')[['enigma_id', 'pangenome_id', 'depot_kos', 'eggnog_kos', 'jaccard']].to_string(index=False))

    all_depot, all_eggnog = set(), set()
    for _, r in matched.iterrows():
        all_depot  |= depot_by_genome.get(r['enigma_genome_id'], set())
        all_eggnog |= eggnog_by_genome.get(r['pangenome_id'], set())
    union = all_depot | all_eggnog
    pool_jac = (len(all_depot & all_eggnog) / len(union)) if union else 0.0
    print(f'\nPooled-set Jaccard (all matched genomes): {pool_jac:.3f}')
else:
    print('Cross-validation skipped (no matched genomes with eggNOG data)')

Per-genome concordance across 153 matched ENIGMA<->pangenome pairs:
  Median Jaccard: 0.140
  Mean   Jaccard: 0.235
  IQR           : 0.057 - 0.400

Top 10 by Jaccard:
 enigma_id       pangenome_id  depot_kos  eggnog_kos  jaccard
      1178 RS_GCF_000146165.2       2111        1758 0.819849
      1202 RS_GCF_003585765.1       1436        1021 0.695652
        41 GB_GCA_004298985.1       1564        1158 0.692786
       124 GB_GCA_004298465.1       1453        1014 0.668019
       118 GB_GCA_004322045.1       2243        1503 0.657522
       215 GB_GCA_004322345.1       1825        1191 0.626753
      1124 RS_GCF_900103195.1       2018        1276 0.618673
         2 RS_GCF_001633105.1       2061        1276 0.614417
       137 GB_GCA_004322135.1       1642        1009 0.605694
       544 RS_GCF_003342755.1       1508         947 0.604575

Pooled-set Jaccard (all matched genomes): 0.777


## 4. Literature Comparison

Compare our cultivation-bias signals against published ORFRC studies.

In [5]:
# Load cultivation coverage results
coverage = pd.read_csv(DATA_DIR / 'ko_cultivation_coverage_full.tsv', sep='\t', index_col=0)
marker_cov = pd.read_csv(DATA_DIR / 'marker_cultivation_coverage.tsv', sep='\t')

# Literature predictions vs our findings
predictions = [
    ('Wood-Ljungdahl', 'depleted', 'Beaver & Neufeld 2024'),
    ('NiFe-hydrogenase', 'depleted', 'Bagnoud 2016'),
    ('Sulfate reduction', 'enriched', 'clay_confined_subsurface'),
    ('Methanogenesis', 'depleted', 'Beaver & Neufeld 2024'),
    ('Denitrification', 'variable', 'Wu 2023'),
    ('N-fixation', 'variable', 'Beaver & Neufeld 2024'),
    ('Metal resistance', 'variable', 'Goff 2024'),
    ('Motility/chemotaxis', 'enriched', 'Cultivation selection'),
    ('Aerobic respiration', 'enriched', 'Cultivation selection'),
    ('Conjugation/MGE', 'depleted', 'Kothari 2019'),
]

print(f'{"Category":<25} {"Predicted":>10} {"Observed log2R":>15} {"Match?":>8} {"Source"}')
print('-' * 80)
for cat, pred, source in predictions:
    sub = marker_cov[marker_cov['category'] == cat]
    if len(sub) > 0:
        mean_lr = sub['log2_ratio'].mean()
        if pred == 'enriched':
            match = 'YES' if mean_lr > 0 else 'NO'
        elif pred == 'depleted':
            match = 'YES' if mean_lr < 0 else 'NO'
        else:
            match = '~'
        print(f'{cat:<25} {pred:>10} {mean_lr:>+15.2f} {match:>8} {source}')
    else:
        print(f'{cat:<25} {pred:>10} {"N/A":>15} {"N/A":>8} {source}')

Category                   Predicted  Observed log2R   Match? Source
--------------------------------------------------------------------------------
Wood-Ljungdahl              depleted           -1.58      YES Beaver & Neufeld 2024
NiFe-hydrogenase            depleted           +6.26       NO Bagnoud 2016
Sulfate reduction           enriched           +0.79      YES clay_confined_subsurface
Methanogenesis              depleted           +0.98       NO Beaver & Neufeld 2024
Denitrification             variable           +2.05        ~ Wu 2023
N-fixation                  variable           +1.56        ~ Beaver & Neufeld 2024
Metal resistance            variable           +3.41        ~ Goff 2024
Motility/chemotaxis         enriched           +7.23      YES Cultivation selection
Aerobic respiration         enriched           +7.29      YES Cultivation selection
Conjugation/MGE             depleted           +3.08       NO Kothari 2019
